# Preprocessing Task

- I left the output of each cell so you can check your solution and know what the expected output should look like.
- If your code is correct but produces slightly different results from mine, that’s totally fine. This rarely happens, but if it does, it will be minor and will be taken into consideration.

# Importing


In [120]:
# Import Needed Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [121]:
# Read data and show first 5 rows
df = pd.read_csv('/content/sample_data/house_prices.csv')
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [122]:
# Data Shape ?
df.shape

(1460, 81)

In [123]:
# Data Info (Data Type of Each column) ?
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

# Handle Null Values

In [124]:
# what the number of Numurical and Categorical Columns ?
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns
print(len(num_cols))
print(len(cat_cols))

38
43


- <b> features to be removed ---> 80%  is nan
- <b> features to filled --------> less than 80 %
- <b> observation to be removed --> less than 3%

In [125]:
df.isnull().sum().sort_values(ascending=False)

,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
...,...
MoSold,0
YrSold,0
SaleType,0
SaleCondition,0


In [126]:
# write function calculate the number of null values in each Feature (column) and Feature has 80% or higher null values drop Feature
# if null count is > 3% and < 80%  fill it by mean
# if null count is < 3% remove this row that contain this null value
def handle_null_values(df):
  null_num = df.isnull().sum()
  for col in df.columns:
    null_percent = null_num[col] / df.shape[0] * 100

    if (null_percent >= 80):
      df.drop(col, axis=1, inplace=True)

    elif (null_percent >= 3 and null_percent < 80):
      if (df[col].dtype == np.number):
        df[col].fillna(df[col].mean(), inplace=True)
      else:
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df.dropna(subset=[col], inplace=True)
  return df

In [127]:
df = handle_null_values(df)
#recheck
df.isnull().sum().sum()

/tmp/ipython-input-1541692353.py:13: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if (df[col].dtype == np.number):
/tmp/ipython-input-1541692353.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipython-input-1541692353.py:13: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly corr

np.int64(0)

# Redundant Handling

### Automated function for both categorical and numircal data

In [128]:
df.shape

(1412, 77)

In [129]:
# write function to calculate the most redundant value in each column how many it repeated ?
def most_redundant(df):
  res = {}
  for col in df.columns:
    if df[col].isnull().all():
       count = 0

    count = df[col].value_counts().iloc[0]
    res[col]=int(count)
  return res

In [130]:
red_count = most_redundant(df)
print(red_count)

{'Id': 1, 'MSSubClass': 515, 'MSZoning': 1111, 'LotFrontage': 251, 'LotArea': 24, 'Street': 1406, 'LotShape': 886, 'LandContour': 1267, 'Utilities': 1411, 'LotConfig': 1015, 'LandSlope': 1335, 'Neighborhood': 215, 'Condition1': 1220, 'Condition2': 1397, 'BldgType': 1189, 'HouseStyle': 693, 'OverallQual': 381, 'OverallCond': 792, 'YearBuilt': 64, 'YearRemodAdd': 165, 'RoofStyle': 1103, 'RoofMatl': 1387, 'Exterior1st': 501, 'Exterior2nd': 491, 'MasVnrType': 1269, 'MasVnrArea': 828, 'ExterQual': 871, 'ExterCond': 1239, 'Foundation': 633, 'BsmtQual': 648, 'BsmtCond': 1301, 'BsmtExposure': 944, 'BsmtFinType1': 426, 'BsmtFinSF1': 426, 'BsmtFinType2': 1246, 'BsmtFinSF2': 1246, 'BsmtUnfSF': 81, 'TotalBsmtSF': 35, 'Heating': 1386, 'HeatingQC': 725, 'CentralAir': 1331, 'Electrical': 1300, '1stFlrSF': 25, '2ndFlrSF': 796, 'LowQualFinSF': 1387, 'GrLivArea': 22, 'BsmtFullBath': 815, 'BsmtHalfBath': 1330, 'FullBath': 738, 'HalfBath': 872, 'BedroomAbvGr': 790, 'KitchenAbvGr': 1361, 'KitchenQual': 705

In [131]:
def handle_redundant(df):
  for col in df.columns:
    if red_count[col] >= df.shape[0] * 0.8:
      df.drop(col, axis=1, inplace=True)
  return df

In [132]:
# if it repeated more than or equal 80% from data rows count remove the feature
# Data row count = data.shape[0]
df = handle_redundant(df)
df.shape

(1412, 47)

# Handling correlation between features

In [133]:
#calculate the correlation matrix  (note the correlation matrix just only for numirical features )
numerical_df = df.select_dtypes(include=['number'])
corr_matrix = numerical_df.corr()
corr_matrix

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,MoSold,YrSold,SalePrice
Id,1.000000,0.010816,-0.012445,-0.034357,-0.031729,0.017332,-0.016396,-0.023834,-0.047418,-0.005680,...,0.026706,-0.017027,-0.003027,0.016094,0.018983,-0.029800,-0.005046,0.026508,-0.004209,-0.024651
MSSubClass,0.010816,1.000000,-0.360014,-0.138298,0.038952,-0.063240,0.032256,0.043906,0.023024,-0.067783,...,0.030657,-0.040128,0.077865,-0.034897,-0.094523,-0.014568,-0.005684,-0.016463,-0.026526,-0.082281
LotFrontage,-0.012445,-0.360014,1.000000,0.305272,0.232351,-0.054635,0.114914,0.078828,0.179330,0.215487,...,0.323671,0.235190,0.061591,0.269574,0.324954,0.075353,0.134402,0.005484,0.006464,0.333477
LotArea,-0.034357,-0.138298,0.305272,1.000000,0.104763,-0.002941,0.014163,0.010765,0.104010,0.213450,...,0.189107,0.268960,-0.025568,0.153812,0.180207,0.172281,0.083860,0.001070,-0.012886,0.264803
OverallQual,-0.031729,0.038952,0.232351,0.104763,1.000000,-0.121155,0.570266,0.540129,0.409656,0.211652,...,0.438254,0.392978,0.518342,0.607663,0.561612,0.230274,0.297678,0.064301,-0.024251,0.786765
OverallCond,0.017332,-0.063240,-0.054635,-0.002941,-0.121155,1.000000,-0.389624,0.057361,-0.134961,-0.057034,...,-0.058852,-0.030047,-0.318043,-0.193904,-0.163341,-0.013206,-0.041823,-0.008530,0.048668,-0.093567
YearBuilt,-0.016396,0.032256,0.114914,0.014163,0.570266,-0.389624,1.000000,0.593163,0.311745,0.242714,...,0.091792,0.143433,0.781869,0.536963,0.475678,0.225469,0.185426,0.008403,-0.014693,0.518736
YearRemodAdd,-0.023834,0.043906,0.078828,0.010765,0.540129,0.057361,0.593163,1.000000,0.172112,0.110541,...,0.189216,0.100277,0.619468,0.423161,0.369665,0.199144,0.218889,0.015944,0.029968,0.500266
MasVnrArea,-0.047418,0.023024,0.179330,0.104010,0.409656,-0.134961,0.311745,0.172112,1.000000,0.259291,...,0.283266,0.241914,0.247464,0.363992,0.370982,0.157566,0.124018,-0.009909,-0.005249,0.474525
BsmtFinSF1,-0.005680,-0.067783,0.215487,0.213450,0.211652,-0.057034,0.242714,0.110541,0.259291,1.000000,...,0.043425,0.250394,0.141824,0.215661,0.288913,0.196236,0.098352,-0.020850,0.020268,0.368849


#### If we have 2 highly correlated features (corr > 0.7), we drop one of them (the variable which is less correlated to the response variable (Output column) )

In [134]:
def drop_high_corr_simple(df, threshold=0.7):
    to_drop = set()

    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if corr_matrix.iloc[i, j] > threshold:
                colname = corr_matrix.columns[i]
                to_drop.add(colname)

    df_new = df.drop(columns=to_drop)

    return df_new, list(to_drop)

In [135]:
df, to_drop = drop_high_corr_simple(df)
print(to_drop)

['SalePrice', 'TotRmsAbvGrd', '1stFlrSF', 'GarageYrBlt', 'GarageArea']


# Handling Outliers
- lower band = q1 - (1.5*IQR)
- Upper band = q3 + (1.5*IQR)

In [136]:
# Write function to remove Outliers that above Upper band or below lower band for each column
def remove_outliers(df):
  df_numeric = df.select_dtypes(include='number')
  for col in df.df_numeric:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_band = q1 - (1.5 * iqr)
    upper_band = q3 + (1.5 * iqr)
    df = df[(df[col] >= lower_band) & (df[col] <= upper_band)]
  return df

In [137]:
def remove_outliers(df):
  df_numeric = df.select_dtypes(include='number')
  for col in df_numeric.columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_band = q1 - (1.5 * iqr)
    upper_band = q3 + (1.5 * iqr)
    df = df[(df[col] >= lower_band) & (df[col] <= upper_band)]
  return df

df = remove_outliers(df)
df.shape

(909, 42)

# Skewness Handling
![image.png](attachment:image.png)


In [138]:
# write function to calc the skewness of each feature
#Note  N = train_data.shape[0]  (rows count)
def calculate_skewness(df):
    df_numeric = df.select_dtypes(include='number')
    N = df_numeric.shape[0]

    skew_dict = {}

    for col in df_numeric.columns:
        x = df_numeric[col].values
        mean = np.mean(x)
        std = np.std(x, ddof=1)  # sample std
        skew = (N / ((N-1)*(N-2))) * np.sum(((x - mean)/std)**3)
        skew_dict[col] = skew

    return pd.Series(skew_dict).sort_values(ascending=False)

df_skew = calculate_skewness(df)
df_skew

,0
MasVnrArea,1.436283
OpenPorchSF,1.170292
WoodDeckSF,0.947834
MSSubClass,0.909159
2ndFlrSF,0.886049
HalfBath,0.847532
OverallCond,0.761246
BsmtUnfSF,0.673012
BsmtFinSF1,0.573608
Fireplaces,0.556985


# Log Transformation
> X = log(1 + | X | )     
this is the equation

In [139]:
# Features that have absolute skewness > 1 do for it Log tranformation
def log_transform_skewed(df, threshold=1):
    skewness = df.select_dtypes(include='number').skew()
    skewed_features = skewness[abs(skewness) > threshold].index
    df[skewed_features] = df[skewed_features].apply(lambda x: np.log1p(np.abs(x)))

    return df, skewed_features

df, transformed_features = log_transform_skewed(df)
transformed_features

Index(['MasVnrArea', 'OpenPorchSF'], dtype='object')

# Transform categorical features

In [140]:
# cat_cols ordinal or nominal?
ordinal_cats = ['BsmtQual', 'LotShape', 'HeatingQC', 'BsmtFinType1',  'ExterQual',
                 'KitchenQual', 'BsmtExposure', 'GarageFinish']
nominal_cats = ['HouseStyle', 'LotConfig', 'RoofStyle', 'GarageType', 'Exterior1st',
                'Foundation', 'MSZoning', 'Exterior2nd', 'Neighborhood','FireplaceQu']

In [141]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# Write code to transform each ordinal and nominal columns
# سيرشك الجميل بقي وشوف كل واحد بيطلع ايه وازاي هتلم الداتا بتاعتك معلش هتعبك معانا يا جميل
# من عييينيييييييييييييييييييييييييي

l_encoder = LabelEncoder()
for col in ordinal_cats:
    df[col] = l_encoder.fit_transform(df[col].astype(str))

df = pd.get_dummies(df, columns=nominal_cats, drop_first=True)

#check after encoding
print(df.dtypes)

Id                        int64
MSSubClass                int64
LotFrontage             float64
LotArea                   int64
LotShape                  int64
                         ...   
Neighborhood_Veenker       bool
FireplaceQu_Fa             bool
FireplaceQu_Gd             bool
FireplaceQu_Po             bool
FireplaceQu_TA             bool
Length: 111, dtype: object


# Transform Numerical features
> I want you apply only Min-Max Scaling for all numerical columns

In [144]:
from sklearn.preprocessing import MinMaxScaler

# write code here
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
scaler = MinMaxScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])
df[num_cols].head()

,Id,MSSubClass,LotFrontage,LotArea,LotShape,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,...,HalfBath,BedroomAbvGr,KitchenQual,Fireplaces,GarageFinish,GarageCars,WoodDeckSF,OpenPorchSF,MoSold,YrSold
0,0.000000,0.4,0.4375,0.400706,1.0,0.625,0.333333,0.949580,0.883333,0.868958,...,0.5,0.666667,0.666667,0.0,0.5,0.666667,0.000000,0.812203,0.090909,0.50
2,0.001372,0.4,0.4750,0.602391,0.0,0.625,0.333333,0.932773,0.866667,0.837797,...,0.5,0.666667,0.666667,0.5,0.5,0.666667,0.000000,0.740189,0.727273,0.50
3,0.002058,0.5,0.3750,0.479939,0.0,0.625,0.333333,0.210084,0.333333,0.000000,...,0.0,0.666667,0.666667,0.5,1.0,1.000000,0.000000,0.705222,0.090909,0.00
4,0.002743,0.4,0.6750,0.819203,0.0,0.750,0.333333,0.924370,0.833333,0.963956,...,0.5,1.000000,0.666667,0.5,0.5,1.000000,0.469438,0.874296,1.000000,0.50
5,0.003429,0.3,0.6875,0.808759,0.0,0.375,0.333333,0.865546,0.750000,0.000000,...,0.5,0.000000,1.000000,0.0,1.0,0.666667,0.097800,0.675795,0.818182,0.75


In [145]:
# print Your final data frame here
# data.head()
df.head()

,Id,MSSubClass,LotFrontage,LotArea,LotShape,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,...,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker,FireplaceQu_Fa,FireplaceQu_Gd,FireplaceQu_Po,FireplaceQu_TA
0,0.000000,0.4,0.4375,0.400706,1.0,0.625,0.333333,0.949580,0.883333,0.868958,...,False,False,False,False,False,False,False,True,False,False
2,0.001372,0.4,0.4750,0.602391,0.0,0.625,0.333333,0.932773,0.866667,0.837797,...,False,False,False,False,False,False,False,False,False,True
3,0.002058,0.5,0.3750,0.479939,0.0,0.625,0.333333,0.210084,0.333333,0.000000,...,False,False,False,False,False,False,False,True,False,False
4,0.002743,0.4,0.6750,0.819203,0.0,0.750,0.333333,0.924370,0.833333,0.963956,...,False,False,False,False,False,False,False,False,False,True
5,0.003429,0.3,0.6875,0.808759,0.0,0.375,0.333333,0.865546,0.750000,0.000000,...,False,False,False,False,False,False,False,True,False,False
